In [1]:
import numpy as np

# X represents a single input feature.
# This range (0 to 10) ensures we see both early rapid change and later saturation, which a linear model cannot fit well.
X = np.linspace(0, 10, 50).reshape(-1, 1)

# The logarithmic shape grows fast initially and slows later. A straight line has only one slope and cannot match both regions simultaneously.
noise = np.random.normal(0, 0.2, size=(50, 1))
y = np.log(X + 1) + noise

In [2]:
'''

1) How many intermediate values (hidden units) do you want?
Ans) We will choose 3.

2) Why more than 1?
Ans) Hidden units allow multiple "pieces" of behavior to bend the curve accurately.

3) Why not too many?
Ans) Too many units increase complexity and instability, while too few cannot bend enough to fit the data.

'''

# Weights affect the shape (slope and bending) of the curve.
# Biases shift the curve's position up or down.
W1 = np.random.uniform(-1, 1, size=(1, 3))
b1 = np.zeros((1, 3))
W2 = np.random.uniform(-1, 1, size=(3, 1))
b2 = np.zeros((1, 1))

In [3]:
def activation(z):
    return np.maximum(0, z)

# Without a slope, the model has no direction signal telling it how to change parameters.
def activation_slope(z):
    return (z > 0).astype(float)

In [4]:
# Epochs: 1000 provides enough iterations for the small learning rate to converge on this dataset.
# Learning Rate: 0.01 is chosen to take small, stable steps without overshooting the optimal weights.
learning_rate = 0.01

for epoch in range(1000):

    # Forward Pass:
    # z_1 = XW_1 + b_1
    z1 = X @ W1 + b1
    # h = activation(z_1)
    h = activation(z1)
    # y_hat = hW_2 + b_2
    y_hat = h @ W2 + b2

    error = y_hat - y

    # Squaring amplifies large mistakes, forcing the model to correct them more aggressively.
    loss = np.mean(error ** 2)

    dL_dy = 2 * error / len(X)

    # These depend on hidden values because the hidden layer's output (h) acts as the input features for the final output layer's weights (W2).
    dL_dW2 = h.T @ dL_dy
    dL_db2 = np.sum(dL_dy, axis=0, keepdims=True)

    dL_dh = dL_dy @ W2.T

    # The activation slope acts as a gate; it zeroes out error flow for negative inputs and passes error flow unchanged for positive inputs.
    dL_dz1 = dL_dh * activation_slope(z1)

    dL_dW1 = X.T @ dL_dz1
    dL_db1 = np.sum(dL_dz1, axis=0, keepdims=True)

    # This connects to the perceptron rule of "moving opposite to the error" by subtracting the gradient scaled by the learning rate.
    W1 -= learning_rate * dL_dW1
    b1 -= learning_rate * dL_db1
    W2 -= learning_rate * dL_dW2
    b2 -= learning_rate * dL_db2

    if epoch % 100 == 0:
        print(f"Epoch {epoch}, Loss: {loss:.4f}")

Epoch 0, Loss: 6.7899
Epoch 100, Loss: 0.0513
Epoch 200, Loss: 0.0426
Epoch 300, Loss: 0.0407
Epoch 400, Loss: 0.0398
Epoch 500, Loss: 0.0391
Epoch 600, Loss: 0.0381
Epoch 700, Loss: 0.0372
Epoch 800, Loss: 0.0364
Epoch 900, Loss: 0.0356
